# Scholar-IR — live demo (track T6: vertical search for science)
**Problem.** Scholarly search should use the structure of papers (title/abstract zones, year, category) and their authority (citations). But raw citation counts grow with age, so mixing them into the ranking buries new, relevant papers. Scholar-IR (1) normalises authority within each publication-year cohort and (2) sets the authority weight λ per query, high for foundational requests and low for recent/specific ones.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
from scholar_ir.config import load_config, p
cfg = load_config()  # config.yaml at the repo root (or $SCHOLAR_IR_CONFIG)
R = p(cfg, "results_dir")

In [ ]:
from scholar_ir.engine import SearchEngine
engine = SearchEngine.load(cfg)
index, papers = engine.index, engine.papers
print(f"{index.N:,} papers, years {papers.year.min()}-{papers.year.max()}, tuned params loaded: {bool(engine.tuned)}")

## 1 · Postings in the title and abstract zone indexes

In [ ]:
term = index.analyzer.terms('transformer')[0]
for z in ('title', 'abstract'):
    zi = index.zone(z); print(f'{z}: df={zi.get_df(term)}  postings (doc, tf, positions):', zi.raw_postings(term, 4))

## 2 · A Boolean + phrase + filter query, with postings touched

In [ ]:
from scholar_ir.boolean_search import boolean_search
from scholar_ir.query_parser import parse
pq = parse('"language model" AND (retrieval OR search) year:>=2022 cat:cs.IR')
docs, st = boolean_search(index, pq)
print(pq.to_string(), '->', len(docs), 'docs; postings entries touched:', st.touched, '; lists:', st.lists)
papers.set_index('doc_id').loc[docs[:5], ['arxiv_id', 'year', 'title']]

## 3 · Same query, four rankers, with score breakdowns

In [ ]:
from scholar_ir.explain import format_explain
Q = 'seminal work on word embeddings'
for s in ('S1', 'S3', 'S4', 'S7'):
    print('=' * 30, s); print(format_explain(engine.search(Q, system=s, k=3, explain=True), max_terms=4))

## 4 · The bias demo: a recent query where raw authority (S4) buries new work and S7 recovers it

In [ ]:
def show(q):
    for s in ('S3', 'S4', 'S7'):
        r = engine.search(q, system=s, k=5)
        lam = f' λ={r.lam:.2f}' if r.lam is not None else ''
        print(f'--- {s}{lam}')
        for x in r.results: print(f'   {x.year}  cites={x.citations:<6} {x.title[:80]}')
show('retrieval-augmented generation for large language models')

In [ ]:
show('survey of recommender systems')

In [ ]:
pd.read_csv(R / 'bias/age_bias_by_system.csv')

## 5 · Natural language → structured query → results → cited answer → citation check

In [ ]:
from scholar_ir.llm_client import LLMClient
from scholar_ir.citation_checker import CitationChecker
from scholar_ir.rag import answer
client = LLMClient(cfg); ck = engine.cfg['checker']; checker = CitationChecker(index, ck['alpha'], ck['tau'])
a = answer('What privacy attacks threaten federated learning? Recent work since 2021 please.', engine, client, checker)
print(a.report())

In [ ]:
ctx = a.context; pid = list(ctx)[0]
for s in [f'Federated learning was invented to train image classifiers on satellites [arXiv:{pid}].',
          'Gradient inversion recovers training images [arXiv:1999.00001].']:
    c = checker.check_sentence(s, ctx); print(s, '->', c.flags, f'support={c.support}')

## 6 · A limitation, live: missing citation data / paraphrase blind spot

In [ ]:
miss = engine.auth[~engine.auth.has_citation_data]
print(f'{len(miss)} of {len(engine.auth)} papers ({100*len(miss)/len(engine.auth):.1f}%) have no OpenAlex match; they get the cohort median g = 0.5')
para = f'A system that finds documents by comparing learned vector encodings of questions and texts [arXiv:{pid}].'
c = checker.check_sentence(para, ctx); print('paraphrase ->', c.flags, f'support={c.support}')

## 7 · Evaluation against the baseline (test split)

In [ ]:
pd.read_csv(R / 'eval/summary_test.csv')

In [ ]:
from IPython.display import Image
Image(str(R / 'eval/ndcg_by_system_test.png'))